# 06 · Long-term memory
**The limitation this fixes:** the checkpointer remembers **one thread**. A new chat is a blank slate.

| Feature | Framework piece |
|---|---|
| Memory across all chats | LangGraph `Store` (`InMemoryStore` here, `PostgresStore` in production) |
| Per-patient isolation | namespace `("memories", patient_token)` from context |
| Semantic recall ("knee" ≈ "joint pain") | Store with an embedding index |
| Tools reach the store | `runtime.store` |
| "Forget me" needs approval | `HumanInTheLoopMiddleware` |

⚠️ This memory is **not the medical record**. It's conversational convenience.

In [ ]:
# Setup: the same cell at the top of every notebook
from dotenv import load_dotenv
load_dotenv("../.env", override=True)   # OPENAI_API_KEY + LANGSMITH_* (set LANGSMITH_TRACING=true to see traces)

MODEL = "openai:gpt-4o-mini"           # swap the provider here, e.g. "anthropic:claude-sonnet-4-6"

## What we built so far

In [ ]:
from common import (CareContext, RAG_PROMPT, get_care_plan, log_reading, log_symptom, get_recent_readings,
                    escalate_to_nurse, dose_guard, pii_middleware, audit_log, ask, resume, show_state)
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

## ❌ The problem: a new thread forgets everything

In [ ]:
agent = create_agent(MODEL, tools=[log_symptom], system_prompt=RAG_PROMPT,
                     context_schema=CareContext, checkpointer=InMemorySaver())
ask(agent, "My knee hurts when I walk to the shop", thread="monday")
ask(agent, "Hi, it's me again. Anything you wanted to follow up on?", thread="friday")

## 1. The Store: a key-value store with namespaces and semantic search

In [ ]:
from langgraph.store.memory import InMemoryStore
from langchain_openai import OpenAIEmbeddings

store = InMemoryStore(index={"embed": OpenAIEmbeddings(model="text-embedding-3-small"), "dims": 1536})

# Try it by hand:
store.put(("demo",), "1", {"fact": "Likes morning appointments"})
store.put(("demo",), "2", {"fact": "Knee pain when walking"})
[(item.value["fact"], round(item.score, 2)) for item in store.search(("demo",), query="joint problems")]

## 2. Memory tools
The namespace comes from context, so a patient can only ever touch **their own** memories.

In [ ]:
from uuid import uuid4
from langchain.tools import tool, ToolRuntime

def my_namespace(runtime):
    return ("memories", runtime.context.patient_token)

@tool
def remember(fact: str, runtime: ToolRuntime[CareContext]) -> str:
    """Save a lasting fact about the patient worth following up next time, e.g. 'knee pain when walking'."""
    runtime.store.put(my_namespace(runtime), str(uuid4()), {"fact": fact})
    return f"Remembered: {fact}"

@tool
def export_my_memories(runtime: ToolRuntime[CareContext]) -> str:
    """Show the patient everything remembered about them."""
    items = runtime.store.search(my_namespace(runtime), limit=100)
    return "\n".join(f"- {i.value['fact']}" for i in items) or "Nothing remembered."

@tool
def forget_me(runtime: ToolRuntime[CareContext]) -> str:
    """Delete everything remembered about the patient. Only when they explicitly ask."""
    items = runtime.store.search(my_namespace(runtime), limit=1000)
    for item in items:
        runtime.store.delete(item.namespace, item.key)
    return f"Deleted {len(items)} memories."

💡 **Is there a prebuilt?** Yes: the **LangMem** library has `create_manage_memory_tool` and `create_search_memory_tool`.
We didn't use them here because LangMem reads the patient from the run config (`config["configurable"]`), not from `CareContext`.
That would break our rule from notebook 01: *the patient's identity only comes from context*.
It also has no "export all" or "forget me" tool, so we'd still write those ourselves.
Our version is still 100% framework: a normal `@tool` + `runtime.store`.

## 3. Load memories into the prompt automatically
Saving is the model's choice (a tool). Recalling is **code**: `@dynamic_prompt` reads the store before every model call.

In [ ]:
from langchain.agents.middleware import dynamic_prompt

@dynamic_prompt
def prompt_with_memories(request):
    ctx = request.runtime.context
    items = request.runtime.store.search(("memories", ctx.patient_token), limit=10)
    facts = "\n".join(f"- {i.value['fact']}" for i in items) or "- nothing yet"
    return (f"{RAG_PROMPT}\n- Use remember to save lasting facts (ongoing problems, preferences).\n"
            f"\nWhat you remember about this patient:\n{facts}\n"
            "If you remember an ongoing problem, ask how it is going.")

## Putting it together: `store=` goes into `create_agent`

In [ ]:
tools = [get_care_plan, log_reading, log_symptom, get_recent_readings, escalate_to_nurse,
         remember, export_my_memories, forget_me]
agent = create_agent(MODEL, tools=tools, context_schema=CareContext,
                     checkpointer=InMemorySaver(), store=store,
                     middleware=[dose_guard, *pii_middleware(), prompt_with_memories, audit_log,
                                 HumanInTheLoopMiddleware(interrupt_on={"escalate_to_nurse": True,
                                                                        "forget_me": True})])

ask(agent, "My knee hurts when I walk to the shop", thread="monday")

🧾 **Receipt:** what's in the store for Sara.

In [ ]:
[item.value for item in store.search(("memories", "p-001"))]

## A brand-new thread still knows

In [ ]:
ask(agent, "Hi, it's me again. Anything you wanted to follow up on?", thread="friday")

## ❌ Failure case: Omar's new chat knows nothing about Sara's knee

In [ ]:
omar = CareContext(user_id="omar", patient_token="p-002")
ask(agent, "What do you remember about me?", thread="omar-1", context=omar)

## "Forget me", with human approval

In [ ]:
ask(agent, "Please forget everything you remember about me.", thread="forget")
resume(agent, {"decisions": [{"type": "approve"}]}, thread="forget")
[item.value for item in store.search(("memories", "p-001"))]   # 🧾 receipt: empty

In [ ]:
ask(agent, "Hi! Anything you wanted to follow up on?", thread="after-forget")

**Expiry:** in production, the Postgres store supports a TTL (configured in `langgraph.json`), so memories expire automatically.

## Where this lives in the backend
- `app/tools.py: `remember`, `export_my_memories`, `forget_me``
- `app/middleware.py: memories in `care_prompt``
- `langgraph.json: the server provides the Store, with the same embedding index`

Same primitives, just split into files. The notebook never imports `app/`.